# 🎨 Generative Adversarial Network (GAN)
## Complete Guide: Theory → Implementation → Sample Project

---

> **GAN** is a framework where two neural networks compete: a **Generator** creates fake data, and a **Discriminator** tries to detect fakes. Through this adversarial game, the Generator learns to produce increasingly realistic data.

### 📚 What You'll Learn
- ✅ Core theory behind GANs
- ✅ How Generator and Discriminator work
- ✅ Training loop and loss functions
- ✅ Complete MNIST Digit Generation project
- ✅ Visualizing generated images epoch by epoch
- ✅ Common pitfalls and how to avoid them

## 📖 Section 1: Theory

### 1.1 The GAN Idea (2014 — Ian Goodfellow)

Imagine a **forger** (Generator) trying to create fake paintings and an **art detective** (Discriminator) trying to spot fakes:

```
Random Noise z ~ N(0,1)
        │
        ▼
  ┌─────────────┐
  │  Generator  │ → Fake Image G(z)
  └─────────────┘
                                    ┌───────────────────┐
Real Image x ──────────────────────►│  Discriminator D  │ → P(real) ∈ [0,1]
Fake Image G(z) ───────────────────►│                   │
                                    └───────────────────┘
```

### 1.2 Mathematical Objective (Minimax Game)

$$\min_G \max_D \; V(D, G) = \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

- **Discriminator maximizes** → correctly classify real as real (D(x)→1) and fake as fake (D(G(z))→0)
- **Generator minimizes** → make Discriminator think fakes are real (D(G(z))→1)

### 1.3 Training Loop

```
For each epoch:
  For each batch:
    Step 1: Train Discriminator
      ├── Sample real images x from dataset
      ├── Sample noise z, generate fake = G(z)
      ├── D_loss = BCE(D(x), 1) + BCE(D(fake), 0)
      └── Backprop + update D weights
    
    Step 2: Train Generator
      ├── Sample new noise z, generate fake = G(z)
      ├── G_loss = BCE(D(fake), 1)  ← trick D into thinking fake is real
      └── Backprop + update G weights (D weights frozen)
```

### 1.4 Key Activation Functions
| Layer | Activation | Why? |
|-------|-----------|------|
| Generator hidden | **ReLU** | Sparse, efficient feature learning |
| Generator output | **Tanh** | Output in [-1,1] matching normalized images |
| Discriminator hidden | **LeakyReLU** | No dead neurons, gradient flows back |
| Discriminator output | **Sigmoid** | Probability output [0,1] |

## ⚙️ Section 2: Setup & Imports

In [ ]:
# Install dependencies (uncomment if running on fresh Colab)
# !pip install tensorflow matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
import os
import time
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist

# Check GPU
print('TensorFlow version:', tf.__version__)
print('GPUs available:', tf.config.list_physical_devices('GPU'))
print('Using:', 'GPU ✅' if tf.config.list_physical_devices('GPU') else 'CPU (consider enabling GPU in Colab)')

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print('\n✅ All imports successful!')

## 🔧 Section 3: Hyperparameters

In [ ]:
# ─────────────────────────────────────────
#  GAN Hyperparameters
# ─────────────────────────────────────────
LATENT_DIM   = 100      # Size of noise vector (z)
IMG_SHAPE    = (28, 28, 1)  # MNIST image shape
IMG_FLAT     = 28 * 28 * 1  # Flattened size = 784

BATCH_SIZE   = 64       # Images per training step
EPOCHS       = 50       # Training epochs (increase for better results)
LR           = 0.0002   # Learning rate (Adam)
BETA_1       = 0.5      # Adam beta_1 (0.5 works better for GANs than 0.9)

SAMPLE_INTERVAL = 10    # Save generated images every N epochs

print('📐 Configuration:')
print(f'  Latent Dimension : {LATENT_DIM}')
print(f'  Image Shape      : {IMG_SHAPE}')
print(f'  Batch Size       : {BATCH_SIZE}')
print(f'  Epochs           : {EPOCHS}')
print(f'  Learning Rate    : {LR}')
print(f'  Adam Beta1       : {BETA_1}')

## 📦 Section 4: Load & Preprocess Data

In [ ]:
# Load MNIST
(X_train, _), (_, _) = mnist.load_data()

print(f'Original shape: {X_train.shape}')  # (60000, 28, 28)

# Preprocess:
# 1. Add channel dimension: (60000, 28, 28) → (60000, 28, 28, 1)
# 2. Normalize from [0, 255] → [-1, 1] to match Tanh output of Generator
X_train = X_train.astype('float32')
X_train = (X_train - 127.5) / 127.5  # → [-1, 1]
X_train = np.expand_dims(X_train, axis=-1)

print(f'Preprocessed shape: {X_train.shape}')
print(f'Value range: [{X_train.min():.1f}, {X_train.max():.1f}]')

# Create TensorFlow dataset
dataset = tf.data.Dataset.from_tensor_slices(X_train)
dataset = dataset.shuffle(buffer_size=60000).batch(BATCH_SIZE, drop_remainder=True)

num_batches = len(X_train) // BATCH_SIZE
print(f'\nTotal samples  : {len(X_train):,}')
print(f'Batch size     : {BATCH_SIZE}')
print(f'Steps per epoch: {num_batches}')

# Visualize real samples
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle('Real MNIST Digits (training data)', fontsize=14, fontweight='bold')
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i, :, :, 0], cmap='gray')
    ax.axis('off')
plt.tight_layout()
plt.show()
print('✅ Data loaded and visualized!')

## 🏗️ Section 5: Build the Generator

The Generator takes a **random noise vector** z and maps it to a **fake image**.

```
z (100,) → Dense(256) → BN → LeakyReLU
         → Dense(512) → BN → LeakyReLU  
         → Dense(1024) → BN → LeakyReLU
         → Dense(784, Tanh) → Reshape(28,28,1)
```

In [ ]:
def build_generator(latent_dim):
    """Generator: Noise → Fake Image"""
    model = keras.Sequential([
        keras.Input(shape=(latent_dim,)),
        
        # Block 1: Expand from noise
        layers.Dense(256),
        layers.BatchNormalization(momentum=0.8),
        layers.LeakyReLU(0.2),
        
        # Block 2: Learn features
        layers.Dense(512),
        layers.BatchNormalization(momentum=0.8),
        layers.LeakyReLU(0.2),
        
        # Block 3: More features
        layers.Dense(1024),
        layers.BatchNormalization(momentum=0.8),
        layers.LeakyReLU(0.2),
        
        # Output: Flatten image with Tanh activation
        layers.Dense(IMG_FLAT, activation='tanh'),
        layers.Reshape(IMG_SHAPE)
    ], name='Generator')
    return model

generator = build_generator(LATENT_DIM)
generator.summary()

# Test: generate a fake image from random noise
test_noise = tf.random.normal([1, LATENT_DIM])
test_output = generator(test_noise, training=False)
print(f'\nTest output shape: {test_output.shape}')  # Should be (1, 28, 28, 1)

plt.figure(figsize=(3, 3))
plt.imshow(test_output[0, :, :, 0], cmap='gray')
plt.title('Untrained Generator Output (random noise)')
plt.axis('off')
plt.show()

## 🔍 Section 6: Build the Discriminator

The Discriminator takes an **image** (real or fake) and outputs a **probability** of it being real.

```
Image(28,28,1) → Flatten(784)
               → Dense(1024) → LeakyReLU → Dropout(0.3)
               → Dense(512)  → LeakyReLU → Dropout(0.3)
               → Dense(256)  → LeakyReLU → Dropout(0.3)
               → Dense(1, Sigmoid) → P(real)
```

In [ ]:
def build_discriminator(img_shape):
    """Discriminator: Image → Real/Fake probability"""
    model = keras.Sequential([
        keras.Input(shape=img_shape),
        
        # Flatten input image
        layers.Flatten(),
        
        # Block 1
        layers.Dense(1024),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),
        
        # Block 2
        layers.Dense(512),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),
        
        # Block 3
        layers.Dense(256),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),
        
        # Output: probability of being REAL
        layers.Dense(1, activation='sigmoid')
    ], name='Discriminator')
    return model

discriminator = build_discriminator(IMG_SHAPE)
discriminator.summary()

# Test: classify the fake image we generated
test_pred = discriminator(test_output, training=False)
print(f'\nUntrained D prediction on fake image: {test_pred.numpy()[0][0]:.4f}')
print('(Random ~0.5 since Discriminator is also untrained)')
print('\n✅ Both networks built!')

## 🔧 Section 7: Loss Functions & Optimizers

In [ ]:
# Binary Cross-Entropy loss
# from_logits=False because we use Sigmoid output
cross_entropy = keras.losses.BinaryCrossentropy(from_logits=False)

# Adam optimizer with GAN-specific beta_1=0.5
# Using 0.5 instead of default 0.9 for more stable GAN training
generator_optimizer     = keras.optimizers.Adam(learning_rate=LR, beta_1=BETA_1)
discriminator_optimizer = keras.optimizers.Adam(learning_rate=LR, beta_1=BETA_1)

def discriminator_loss(real_output, fake_output):
    """
    Discriminator wants:
      - D(real) → 1 (classify real as real)
      - D(fake) → 0 (classify fake as fake)
    """
    # Label smoothing: use 0.9 instead of 1.0 to prevent overconfidence
    real_labels = tf.ones_like(real_output) * 0.9  # Soft labels
    fake_labels = tf.zeros_like(fake_output)
    
    real_loss = cross_entropy(real_labels, real_output)
    fake_loss = cross_entropy(fake_labels, fake_output)
    return real_loss + fake_loss

def generator_loss(fake_output):
    """
    Generator wants:
      - D(G(z)) → 1 (fool discriminator into thinking fake is real)
    """
    return cross_entropy(tf.ones_like(fake_output), fake_output)

print('Loss Functions defined:')
print('  Discriminator Loss = BCE(D(real), 1) + BCE(D(fake), 0)')
print('  Generator Loss     = BCE(D(G(z)), 1)')
print('\nOptimizers: Adam(lr=0.0002, beta_1=0.5)')
print('\n✅ Loss functions and optimizers ready!')

## 🏋️ Section 8: Training Step (Core GAN Loop)

In [ ]:
@tf.function  # Compile to TF graph for speed
def train_step(real_images):
    """
    One training step:
    1. Generate fake images
    2. Train Discriminator on real + fake
    3. Train Generator to fool Discriminator
    """
    batch_size = tf.shape(real_images)[0]
    
    # Sample random noise
    noise = tf.random.normal([batch_size, LATENT_DIM])
    
    # ── Step 1: Train Discriminator ──────────────────────────────
    with tf.GradientTape() as disc_tape:
        fake_images = generator(noise, training=True)
        
        real_output = discriminator(real_images, training=True)
        fake_output = discriminator(fake_images, training=True)
        
        d_loss = discriminator_loss(real_output, fake_output)
    
    # Apply gradients to Discriminator
    d_gradients = disc_tape.gradient(d_loss, discriminator.trainable_variables)
    discriminator_optimizer.apply_gradients(
        zip(d_gradients, discriminator.trainable_variables)
    )
    
    # ── Step 2: Train Generator ──────────────────────────────────
    noise = tf.random.normal([batch_size, LATENT_DIM])  # Fresh noise
    
    with tf.GradientTape() as gen_tape:
        fake_images = generator(noise, training=True)
        fake_output = discriminator(fake_images, training=False)  # D frozen
        g_loss = generator_loss(fake_output)
    
    # Apply gradients to Generator only
    g_gradients = gen_tape.gradient(g_loss, generator.trainable_variables)
    generator_optimizer.apply_gradients(
        zip(g_gradients, generator.trainable_variables)
    )
    
    return d_loss, g_loss

print('✅ Training step compiled!')
print('Each step:')
print('  1. Sample noise → Generator → Fake images')
print('  2. Train Discriminator: Real=1, Fake=0')
print('  3. Train Generator: Fake=1 (fool D)')

## 📸 Section 9: Visualization Helper

In [ ]:
# Fixed noise to track progress of same samples across epochs
FIXED_NOISE = tf.random.normal([16, LATENT_DIM])

def plot_generated_images(epoch, noise=FIXED_NOISE, save=False):
    """Plot a 4x4 grid of generated images."""
    generated = generator(noise, training=False)
    
    fig, axes = plt.subplots(4, 4, figsize=(8, 8))
    fig.suptitle(f'GAN Generated Images — Epoch {epoch}', 
                 fontsize=14, fontweight='bold', y=1.01)
    fig.patch.set_facecolor('#1a1a2e')
    
    for i, ax in enumerate(axes.flat):
        # Rescale from [-1, 1] → [0, 1] for display
        img = (generated[i, :, :, 0].numpy() + 1) / 2.0
        ax.imshow(img, cmap='plasma')
        ax.axis('off')
    
    plt.tight_layout()
    if save:
        plt.savefig(f'generated_epoch_{epoch:03d}.png', 
                    dpi=100, bbox_inches='tight')
    plt.show()
    plt.close()

print('Showing initial Generator output (before training):')
plot_generated_images(0)

## 🚀 Section 10: Full Training Loop

In [ ]:
# Lists to record losses
g_losses = []
d_losses = []
epoch_list = []

print('🚀 Starting GAN Training...')
print('=' * 60)
print(f'  Epochs      : {EPOCHS}')
print(f'  Batch size  : {BATCH_SIZE}')
print(f'  Steps/epoch : {num_batches}')
print('=' * 60)

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_d_losses = []
    epoch_g_losses = []
    
    for real_batch in dataset:
        d_loss, g_loss = train_step(real_batch)
        epoch_d_losses.append(float(d_loss))
        epoch_g_losses.append(float(g_loss))
    
    # Compute epoch averages
    avg_d = np.mean(epoch_d_losses)
    avg_g = np.mean(epoch_g_losses)
    g_losses.append(avg_g)
    d_losses.append(avg_d)
    epoch_list.append(epoch)
    
    # Print progress every 5 epochs
    if epoch % 5 == 0 or epoch == 1:
        elapsed = time.time() - start_time
        print(f'Epoch [{epoch:3d}/{EPOCHS}] | '
              f'D_loss: {avg_d:.4f} | '
              f'G_loss: {avg_g:.4f} | '
              f'Time: {elapsed:.0f}s')
    
    # Show generated images periodically
    if epoch % SAMPLE_INTERVAL == 0:
        plot_generated_images(epoch, save=True)

total_time = time.time() - start_time
print('=' * 60)
print(f'✅ Training complete! Total time: {total_time:.0f}s ({total_time/60:.1f} min)')

## 📊 Section 11: Visualize Training Results

In [ ]:
# ── Plot 1: Loss Curves ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('GAN Training Dynamics', fontsize=16, fontweight='bold')

axes[0].plot(epoch_list, g_losses, color='#e63946', label='Generator Loss', linewidth=2)
axes[0].plot(epoch_list, d_losses, color='#457b9d', label='Discriminator Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Generator vs Discriminator Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_facecolor('#f8f9fa')

# Smoothed loss
window = max(1, EPOCHS // 10)
g_smooth = np.convolve(g_losses, np.ones(window)/window, mode='valid')
d_smooth = np.convolve(d_losses, np.ones(window)/window, mode='valid')

axes[1].plot(g_smooth, color='#e63946', label='Generator (smoothed)', linewidth=2)
axes[1].plot(d_smooth, color='#457b9d', label='Discriminator (smoothed)', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss (smoothed)')
axes[1].set_title('Smoothed Loss Curves')
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_facecolor('#f8f9fa')

plt.tight_layout()
plt.savefig('gan_loss_curves.png', dpi=100, bbox_inches='tight')
plt.show()
print('📊 Loss curves saved as gan_loss_curves.png')

In [ ]:
# ── Plot 2: Real vs Generated Comparison ───────────────────────
fig, axes = plt.subplots(2, 8, figsize=(18, 5))
fig.suptitle('🔴 REAL vs 🟢 GENERATED Images', 
             fontsize=14, fontweight='bold')
fig.patch.set_facecolor('#0d0d0d')

# Row 1: Real MNIST
for i in range(8):
    img = (X_train[i, :, :, 0] + 1) / 2.0
    axes[0, i].imshow(img, cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title('REAL', color='#ff6b6b', fontsize=11, pad=5)

# Row 2: Generated
gen_noise = tf.random.normal([8, LATENT_DIM])
generated = generator(gen_noise, training=False)
for i in range(8):
    img = (generated[i, :, :, 0].numpy() + 1) / 2.0
    axes[1, i].imshow(img, cmap='plasma')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title('GENERATED', color='#69db7c', fontsize=11, pad=5)

plt.tight_layout()
plt.savefig('real_vs_generated.png', dpi=100, bbox_inches='tight', 
            facecolor='#0d0d0d')
plt.show()
print('🖼️ Comparison saved as real_vs_generated.png')

## 🧪 Section 12: Latent Space Exploration

Explore how the Generator transforms different noise vectors.

In [ ]:
# ── Interpolate between two points in latent space ─────────────
# This shows smooth transition between two generated images

z1 = tf.random.normal([1, LATENT_DIM])
z2 = tf.random.normal([1, LATENT_DIM])

n_steps = 10
interpolated = []
for alpha in np.linspace(0, 1, n_steps):
    z_interp = (1 - alpha) * z1 + alpha * z2  # Linear interpolation
    img = generator(z_interp, training=False)
    interpolated.append(img[0, :, :, 0].numpy())

fig, axes = plt.subplots(1, n_steps, figsize=(20, 2.5))
fig.suptitle('🔍 Latent Space Interpolation (z1 → z2)', 
             fontsize=13, fontweight='bold')

for i, (ax, img) in enumerate(zip(axes, interpolated)):
    img_display = (img + 1) / 2.0
    ax.imshow(img_display, cmap='plasma')
    ax.axis('off')
    ax.set_title(f'α={i/(n_steps-1):.1f}', fontsize=8)

plt.tight_layout()
plt.savefig('latent_space_interpolation.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Latent space interpolation shows smooth transitions between generated images')
print('This demonstrates the Generator has learned a continuous latent space!')

## 📋 Section 13: Summary & Key Takeaways

### What We Built
| Component | Details |
|-----------|--------|
| **Generator** | 3-layer MLP → 784 outputs → 28×28 images |
| **Discriminator** | 3-layer MLP → 1 output (real/fake probability) |
| **Dataset** | MNIST 60K handwritten digits |
| **Loss** | Binary Cross-Entropy (with label smoothing) |
| **Optimizer** | Adam (lr=0.0002, β₁=0.5) |

### Key GAN Training Tips
1. 🔶 **LeakyReLU** in Discriminator (not regular ReLU) — prevents dead neurons
2. 🔶 **BatchNormalization** in Generator — stabilizes training
3. 🔶 **Label Smoothing** (0.9 not 1.0) — prevents Discriminator overconfidence
4. 🔶 **Adam β₁=0.5** instead of 0.9 for more stability
5. 🔶 **Separate optimizers** for G and D
6. 🔶 **Don't train one network too much** before the other

### GAN Training Signals (What Good Training Looks Like)
- D_loss ≈ 1.0–1.4 (not 0 or very high)
- G_loss decreasing over time
- Both losses oscillate but stabilize

### Next Steps
- 🚀 Try **DCGAN** (replace Dense with Conv2D layers)
- 🚀 Try **Conditional GAN** (generate specific digits)
- 🚀 Try **WGAN** (Wasserstein loss for more stable training)
- 🚀 Try on **CelebA** dataset for face generation

In [ ]:
print('=' * 60)
print('         🎨  GAN TRAINING COMPLETE  🎨')
print('=' * 60)
print(f'Model: Vanilla GAN on MNIST')
print(f'Epochs trained    : {EPOCHS}')
print(f'Final G Loss      : {g_losses[-1]:.4f}')
print(f'Final D Loss      : {d_losses[-1]:.4f}')
print(f'Training time     : {total_time:.0f}s')
print()
print('Files saved:')
print('  📊 gan_loss_curves.png')
print('  🖼️  real_vs_generated.png')
print('  🔍  latent_space_interpolation.png')
print('  📸  generated_epoch_*.png (every 10 epochs)')
print()
print('Next → Try DCGAN, WGAN, or Conditional GAN!')
print('=' * 60)